# Data cleaning visualization

This notebook documents what was pulled from VDJdb, IEDB, and McPAS-TCR, how each dataset was cleaned, what remained after each step, and what the final pooled TRB dataset looks like.

## Final outputs

- `processed/vdjdb_trb_clean.csv`
- `processed/iedb_trb_clean.csv`
- `processed/mcpas_trb_clean.csv`
- `processed/combined_trb_clean.csv`

All cleaned outputs use the same standardized columns:

- `cdr3`
- `tcr_chain`
- `v_gene`
- `j_gene`
- `peptide`
- `mhc_a`
- `mhc_b`
- `mhc_class`
- `source`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_VDJDB = ROOT / "vdjdb_csv" / "vdjdb.csv"
RAW_IEDB = ROOT / "iedb_data" / "tcr_full_v3.csv"
RAW_MCPAS = ROOT / "McPAS-TCR.csv"
VDJDB_CLEAN = ROOT / "processed" / "vdjdb_trb_clean.csv"
IEDB_CLEAN = ROOT / "processed" / "iedb_trb_clean.csv"
MCPAS_CLEAN = ROOT / "processed" / "mcpas_trb_clean.csv"
COMBINED_CLEAN = ROOT / "processed" / "combined_trb_clean.csv"
FIG_DIR = ROOT / "processed" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ROOT

In [ ]:
vdj_raw = pd.read_csv(RAW_VDJDB, low_memory=False)
iedb_raw = pd.read_csv(RAW_IEDB, header=[0, 1], low_memory=False)
mcpas_raw = pd.read_csv(RAW_MCPAS, na_values=["NA"], keep_default_na=True, low_memory=False)
vdj_clean = pd.read_csv(VDJDB_CLEAN, low_memory=False)
iedb_clean = pd.read_csv(IEDB_CLEAN, low_memory=False)
mcpas_clean = pd.read_csv(MCPAS_CLEAN, low_memory=False)
combined_clean = pd.read_csv(COMBINED_CLEAN, low_memory=False)

print("vdj_raw", vdj_raw.shape)
print("iedb_raw", iedb_raw.shape)
print("mcpas_raw", mcpas_raw.shape)
print("vdj_clean", vdj_clean.shape)
print("iedb_clean", iedb_clean.shape)
print("mcpas_clean", mcpas_clean.shape)
print("combined_clean", combined_clean.shape)

## Source columns pulled from each database

### Table: VDJdb raw-to-standardized column map

In [ ]:
vdjdb_column_map = pd.DataFrame(
    [
        ["cdr3", "cdr3", "TCR beta CDR3 amino-acid sequence"],
        ["gene", "tcr_chain", "TCR chain label, filtered to TRB"],
        ["v.segm", "v_gene", "V gene"],
        ["j.segm", "j_gene", "J gene"],
        ["antigen.epitope", "peptide", "Epitope / peptide sequence"],
        ["mhc.a", "mhc_a", "Primary HLA / MHC field"],
        ["mhc.b", "mhc_b", "Secondary MHC-associated field, often B2M"],
        ["mhc.class", "mhc_class", "MHC class label"],
        ["species", "filter only", "Used to keep HomoSapiens only"],
    ],
    columns=["VDJdb raw column", "Standardized column", "Use"],
)
vdjdb_column_map

### Table: Head of raw VDJdb data

In [ ]:
vdj_raw.head()

### Table: IEDB raw-to-standardized column map

In [ ]:
iedb_column_map = pd.DataFrame(
    [
        ["('Chain 2', 'CDR3 Curated') or ('Chain 2', 'CDR3 Calculated')", "cdr3", "TCR beta CDR3 amino-acid sequence"],
        ["constant TRB", "tcr_chain", "Chain label assigned during standardization"],
        ["('Chain 2', 'Curated V Gene') or ('Chain 2', 'Calculated V Gene')", "v_gene", "V gene"],
        ["('Chain 2', 'Curated J Gene') or ('Chain 2', 'Calculated J Gene')", "j_gene", "J gene"],
        ["('Epitope', 'Name')", "peptide", "Epitope / peptide sequence"],
        ["('Assay', 'MHC Allele Names')", "mhc_a", "Main MHC allele field"],
        ["not available as a separate comparable field", "mhc_b", "Left blank in IEDB cleaned output"],
        ["inferred from ('Assay', 'MHC Allele Names')", "mhc_class", "Inferred MHCI vs MHCII"],
        ["('Chain 2', 'Type') and ('Chain 2', 'Organism IRI')", "filter only", "Used to keep human beta-chain TCR rows"],
    ],
    columns=["IEDB raw column(s)", "Standardized column", "Use"],
)
iedb_column_map

### Table: Head of raw IEDB data

In [ ]:
iedb_raw.head()

### Table: McPAS raw-to-standardized column map

In [ ]:
mcpas_column_map = pd.DataFrame(
    [
        ["CDR3.beta.aa", "cdr3", "TCR beta CDR3 amino-acid sequence"],
        ["constant TRB", "tcr_chain", "Chain label assigned during standardization"],
        ["TRBV", "v_gene", "V gene"],
        ["TRBJ", "j_gene", "J gene"],
        ["Epitope.peptide", "peptide", "Epitope / peptide sequence"],
        ["MHC", "mhc_a", "Main MHC allele field"],
        ["not available as a separate comparable field", "mhc_b", "Left blank in McPAS cleaned output"],
        ["inferred from MHC", "mhc_class", "Inferred MHCI vs MHCII"],
        ["Species", "filter only", "Used to keep Human only"],
    ],
    columns=["McPAS raw column", "Standardized column", "Use"],
)
mcpas_column_map

### Table: Head of raw McPAS data

In [ ]:
mcpas_raw.head()

## Cleaning steps

### Table: VDJdb cleaning steps

In [ ]:
vdj_steps = pd.DataFrame(
    [
        ["1", "Keep only HomoSapiens rows", "species == 'HomoSapiens'"],
        ["2", "Keep only TRB rows", "gene == 'TRB'"],
        ["3", "Require non-empty fields", "cdr3, v.segm, j.segm, mhc.a, mhc.b, mhc.class, antigen.epitope"],
        ["4", "Remove wildcard sequences", "drop rows where cdr3 or antigen.epitope contains X, * or ?"],
        ["5", "Remove multi-peptide TCRs", "drop TCR identities mapping to more than one peptide"],
        ["6", "Drop duplicate rows", "deduplicate on the standardized biological fields"],
    ],
    columns=["Step", "Operation", "Rule"],
)
vdj_steps

### Table: IEDB cleaning steps

In [ ]:
iedb_steps = pd.DataFrame(
    [
        ["1", "Keep only beta-chain rows", "('Chain 2', 'Type') == 'beta'"],
        ["2", "Keep only human rows", "('Chain 2', 'Organism IRI') contains NCBITaxon_9606"],
        ["3", "Standardize columns", "use curated CDR3/V/J when present, otherwise calculated values"],
        ["4", "Require non-empty fields", "cdr3, v_gene, j_gene, peptide, mhc_a, mhc_class"],
        ["5", "Remove wildcard sequences", "drop rows where cdr3 or peptide contains X, * or ?"],
        ["6", "Remove multi-peptide TCRs", "drop TCR identities mapping to more than one peptide"],
        ["7", "Drop duplicate rows", "deduplicate on the standardized biological fields"],
    ],
    columns=["Step", "Operation", "Rule"],
)
iedb_steps

### Table: McPAS cleaning steps

In [ ]:
mcpas_steps = pd.DataFrame(
    [
        ["1", "Keep only Human rows", "Species == 'Human'"],
        ["2", "Standardize columns", "use CDR3.beta.aa, TRBV, TRBJ, Epitope.peptide, and MHC"],
        ["3", "Infer mhc_class", "infer MHCI vs MHCII from the MHC field"],
        ["4", "Require non-empty fields", "cdr3, v_gene, j_gene, peptide, mhc_a, mhc_class"],
        ["5", "Remove wildcard sequences", "drop rows where cdr3 or peptide contains X, * or ?"],
        ["6", "Remove multi-peptide TCRs", "drop TCR identities mapping to more than one peptide"],
        ["7", "Drop duplicate rows", "deduplicate on the standardized biological fields"],
    ],
    columns=["Step", "Operation", "Rule"],
)
mcpas_steps

## VDJdb retention summary

### Table: VDJdb rows retained and removed at each step

In [ ]:
vdj_counts = pd.DataFrame(
    [
        ["Loaded raw VDJdb rows", 226494, 0],
        ["After HomoSapiens filter", 206106, 20388],
        ["After TRB filter", 113281, 92825],
        ["After required-field filter", 113281, 0],
        ["After wildcard filter", 113281, 0],
        ["After multi-peptide filter", 101798, 11483],
        ["After de-duplication", 83644, 18154],
    ],
    columns=["Stage", "Rows remaining", "Rows removed at step"],
)
vdj_counts

In [ ]:
ax = vdj_counts.set_index("Stage")["Rows remaining"].plot(kind="bar", figsize=(10, 4), color="#4C78A8", title="VDJdb rows remaining after each cleaning step")
ax.set_ylabel("rows")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "vdjdb_cleaning_steps.png", dpi=200, bbox_inches="tight")
plt.show()

## IEDB retention summary

### Table: IEDB rows retained and removed at each step

In [ ]:
iedb_counts = pd.DataFrame(
    [
        ["Loaded raw IEDB rows", 226280, 0],
        ["After beta-chain filter", 195028, 31252],
        ["After human filter", 187488, 7540],
        ["After required-field filter", 160630, 26858],
        ["After wildcard filter", 160626, 4],
        ["After multi-peptide filter", 146819, 13807],
        ["After de-duplication", 144443, 2376],
    ],
    columns=["Stage", "Rows remaining", "Rows removed at step"],
)
iedb_counts

In [ ]:
ax = iedb_counts.set_index("Stage")["Rows remaining"].plot(kind="bar", figsize=(10, 4), color="#59A14F", title="IEDB rows remaining after each cleaning step")
ax.set_ylabel("rows")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "iedb_cleaning_steps.png", dpi=200, bbox_inches="tight")
plt.show()

## McPAS retention summary

### Table: McPAS rows retained and removed at each step

In [ ]:
mcpas_counts = pd.DataFrame(
    [
        ["Loaded raw McPAS rows", 40731, 0],
        ["After Human filter", 36474, 4257],
        ["After required-field filter", 9370, 27104],
        ["After wildcard filter", 9361, 9],
        ["After multi-peptide filter", 8764, 597],
        ["After de-duplication", 8268, 496],
    ],
    columns=["Stage", "Rows remaining", "Rows removed at step"],
)
mcpas_counts

In [ ]:
ax = mcpas_counts.set_index("Stage")["Rows remaining"].plot(kind="bar", figsize=(10, 4), color="#EDC948", title="McPAS rows remaining after each cleaning step")
ax.set_ylabel("rows")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "mcpas_cleaning_steps.png", dpi=200, bbox_inches="tight")
plt.show()

## What remained after pooling

### Table: Pooled dataset row counts

In [ ]:
pooled_summary = pd.DataFrame(
    [
        ["VDJdb cleaned rows", 83644],
        ["IEDB cleaned rows", 144443],
        ["McPAS cleaned rows", 8268],
        ["Sum before pooling", 236355],
        ["Exact cross-source duplicates removed during pooling", 0],
        ["Final combined rows", 236355],
    ],
    columns=["Quantity", "Value"],
)
pooled_summary

### Table: Final combined rows by source

In [ ]:
combined_clean["source"].value_counts().rename_axis("source").reset_index(name="rows")

## What the combined dataset looks like

### Table: Combined dataset summary statistics

In [ ]:
combined_summary = pd.Series(
    {
        "rows": len(combined_clean),
        "unique CDR3s": combined_clean["cdr3"].nunique(),
        "unique peptides": combined_clean["peptide"].nunique(),
        "unique V genes": combined_clean["v_gene"].nunique(),
        "unique J genes": combined_clean["j_gene"].nunique(),
        "unique mhc_a": combined_clean["mhc_a"].nunique(),
        "rows from VDJdb": int((combined_clean["source"] == "VDJdb").sum()),
        "rows from IEDB": int((combined_clean["source"] == "IEDB").sum()),
        "rows from McPAS": int((combined_clean["source"] == "McPAS").sum()),
    }
)
combined_summary

In [ ]:
combined_clean = combined_clean.assign(
    peptide_len=combined_clean["peptide"].astype(str).str.len(),
    cdr3_len=combined_clean["cdr3"].astype(str).str.len(),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
combined_clean["peptide_len"].plot(kind="hist", bins=20, ax=axes[0], color="#F28E2B", title="Combined peptide length")
combined_clean["cdr3_len"].plot(kind="hist", bins=25, ax=axes[1], color="#E15759", title="Combined CDR3 length")
axes[0].set_xlabel("length")
axes[1].set_xlabel("length")
plt.tight_layout()
plt.savefig(FIG_DIR / "combined_length_distributions.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 16))
combined_clean["source"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#4E79A7", title="Rows by source")
combined_clean["mhc_a"].value_counts().head(20).sort_values().plot(kind="barh", ax=axes[1], color="#76B7B2", title="Top 20 mhc_a values")
combined_clean["v_gene"].value_counts().head(20).sort_values().plot(kind="barh", ax=axes[2], color="#59A14F", title="Top 20 V genes")
combined_clean["j_gene"].value_counts().head(20).sort_values().plot(kind="barh", ax=axes[3], color="#B07AA1", title="Top 20 J genes")
plt.tight_layout()
plt.savefig(FIG_DIR / "combined_top_features.png", dpi=200, bbox_inches="tight")
plt.show()

### Table: Example rows from the final combined dataset

In [ ]:
combined_clean.head()